# Deterministic and Rule-Based Reward Functions

---

TRL-transformer reinforcement learning

---

Now that we know how advantages are calculated from rewards, we need to design the reward functions themselves. In modern RLVR (Reinforcement Learning with Verifiable Rewards), reward functions are deterministic Python functions, not black-box neural networks.

---

## 1. Anatomy of a Verifiable Reward Function

A reward function in modern frameworks such as TRL’s GRPOTrainer is a Python callable that takes the completions and dataset metadata and returns a list of scalar floats:

$$
\text{reward\_func}(\text{completions}, \text{prompts}, **\text{metadata}) \to [r_1, r_2, \dots, r_N]
$$

There are three main types of deterministic reward components:

```text
Composite Reward Signal
├── Format Checker
├── Syntax / Parser
└── Execution Verifier
```

### A. Format / reasoning-tag reward

This ensures the model adheres to structured output contracts, such as generating internal reasoning before the final answer.

- Mechanism: regex matching on delimiters like `<think>...</think>` and `<answer>...</answer>`
- Example:
  - both opening and closing tags present with non-empty content: $+0.2$
  - malformed tags, such as a missing closing tag: $0.0$

### B. Syntactic / static-analysis reward

This verifies that the output can be parsed without runtime crashes.

- Mechanism: for code, run `ast.parse(code_str)`; for JSON, run `json.loads(json_str)`
- Example:
  - parses successfully: $+0.1$
  - syntax error or JSON decode error: $0.0$

### C. Deterministic execution / ground-truth reward

This is the core correctness signal.

- Math: extract the candidate answer and parse symbolic equality with SymPy, such as $\frac{1}{2} = 0.5$
- Coding: execute the generated code against a suite of input/output assertions inside an isolated runner or sandbox
- Example:
  - all assertions pass: $+1.0$
  - any assertion fails or times out: $0.0$

## 2. The Mathematics of Combining Rewards

In TRL, you can pass multiple reward functions. The total reward is the weighted sum:

$$
R_{\text{total}} = w_{\text{format}} R_{\text{format}} + w_{\text{syntax}} R_{\text{syntax}} + w_{\text{correctness}} R_{\text{correctness}}
$$

## 3. Engineering Traps and Best Practices in Reward Design

### Reward scale imbalance

- Bad setup: $R_{\text{format}} = 1.0$ and $R_{\text{correctness}} = 0.5$
- Result: the model will ignore solving the problem and focus purely on formatting

- Good setup: $R_{\text{format}} = 0.1$ and $R_{\text{correctness}} = 1.0$
- Result: formatting is a minor bonus and correctness dominates the gradient direction

### Strict timeouts and sandboxing for code execution

Model-generated code can contain infinite loops such as `while True: pass` or malicious calls such as `os.system("rm -rf /")`.

Verifiers must use strict execution timeouts, for example a maximum of 1.0 second per test, and run in isolated subprocesses or sandboxes.

### Soft penalties vs. hard drops

If a solution is correct but violates formatting slightly, dropping the reward completely to $0.0$ can destroy valuable reasoning signal. Modular reward components allow the model to receive $+1.0$ for math correctness while receiving $0.0$ for format, learning both independently.


## Execution / Correctness Check

Exact string matching against a reference query is often a poor reward signal because two different SQL statements can be semantically identical.

### Example

```sql
-- Query A
SELECT id, name FROM users WHERE age > 21;

-- Query B (equivalent logic, different syntax)
SELECT name, id FROM users WHERE 21 < age;
```

### Robust execution check

1. Spin up an in-memory SQLite or DuckDB test database loaded with representative mock tables.
2. Execute the model’s generated query:
   ```python
   result_pred = db.execute(model_query).fetchall()
   ```
3. Execute the reference query on the exact same database:
   ```python
   result_gold = db.execute(gold_query).fetchall()
   ```
4. Compare the resulting data tables.
   - If `set(result_pred) == set(result_gold)`, assign a reward of $+1.0$.
